# Lab24. 시계열·통계 시각화

PPT를 보며 각 코드 셀의 `____`를 채우세요. 밑줄 전체를 지우고 값 또는 이름을 입력한 뒤 실행합니다. 문자열은 따옴표까지 입력하세요. 위에서부터 순서대로 실행합니다.

## 0. 한글 표시 준비

`____`에 `koreanize-matplotlib`을 입력하고 실행하세요.

In [ ]:
!pip install ____

## 1. 자료 불러오기

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import koreanize_matplotlib

BASE = 'https://raw.githubusercontent.com/hongsukyi/Lectures/main/data/'
from pathlib import Path

source = 'vds_clean.csv' if Path('vds_clean.csv').exists() else BASE + 'vds_clean.csv'
df = pd.read_csv(source, encoding='utf-8-sig', parse_dates=[____]) 

df.head(2)

## 2. 날짜 색인과 집계자료 만들기

In [ ]:
L = ['새벽(0-5)', '오전(6-11)', '오후(12-17)', '저녁(18-23)']
df['time_4bin'] = pd.Categorical(df['time_4bin'], categories=L, ordered=True)
df['day'] = df['Date'].dt.normalize()

daily = (df.groupby(['day', 'time_4bin'], observed=True, as_index=False)
    .agg(Speed=('Speed', 'mean'), ToVol=('ToVol', 'sum'),
         OccRate=('OccRate', 'mean'), is_weekend=('is_weekend', 'first')))
daily['구분'] = daily['is_weekend'].map({False: '평일', True: '주말'})

time_df = df.sort_values('Date').set_index('Date')
daily_mean = (time_df.resample(____)
    .agg(Speed=('Speed','mean'),OccRate=('OccRate','mean'),ToVol=('ToVol','sum')).reset_index()) 

print('5분 자료:', df.shape, '/ 날짜별 집계:', daily_mean.shape)

## 3. 5분 속도와 1시간 이동평균

In [ ]:
part = time_df.loc['2017-04-03'].copy()
part['Speed_1h'] = part['Speed'].rolling(____).mean() 

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(part.index, part.Speed, alpha=.3, 
        label='5분 속도')
ax.plot(part.index, part.Speed_1h, lw=2.5, 
        label='1시간 이동평균')
ax.legend()
ax.set_title('5분 속도와 1시간 이동평균 (2017-04-03)')
plt.show()

## 4. 평일·주말 24시간 패턴

In [ ]:
plot_df = df.assign(구분=df.is_weekend.map( 
    {False: '평일', True: '주말'}))
fig, ax = plt.subplots(figsize=(5, 3))
sns.lineplot(data=plot_df, x='hour', 
             y='Speed', hue=____, 
             errorbar=None, marker='o')
plt.xticks(range(0, 24, 2))
plt.title('평일·주말 24시간 평균 속도')
plt.show()

## 5. 분포·관측점·평균 겹쳐 그리기

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
sns.violinplot(data=daily, x='time_4bin', 
               y='Speed', order=L, inner=None, 
               cut=0,hue='time_4bin', 
               legend=False, alpha=.45, ax=ax)
sns.stripplot(data=daily, x='time_4bin', 
              y='Speed', order=L, color='black', 
              alpha=.3, ax=ax)
sns.pointplot(data=daily, x='time_4bin', 
              y='Speed', order=L, color='C1',
              errorbar=('ci', ____), 
              capsize=.1, ax=ax)
plt.title('시간대별 속도: 분포 + 실제값 + 평균') 
plt.show()

## 6. 요일×시각 속도 히트맵

In [ ]:
names = ['월', '화', '수', '목', '금', '토', '일']
h = df.pivot_table(index='dow', columns='hour', 
                   values=____, aggfunc='mean' 
                  ).reindex(range(7))
h.index = names
fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(h, cmap='YlGnBu_r', 
            linewidths=.25, 
            cbar_kws={'label': '평균 속도'})
plt.title('요일×시각 평균 속도')
plt.show()

## 7. 재현 가능한 Bootstrap 신뢰구간

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
sns.pointplot(data=daily, x='time_4bin', 
              y='Speed', order=L,
              errorbar=('ci', 95), 
              n_boot=2000, seed=____, 
              capsize=.12)
plt.title('Bootstrap 95% 신뢰구간')
plt.xticks(rotation=15)
plt.show()

## 8. 평일·주말 속도 차이 검정

In [ ]:
day_avg = df.groupby('day').agg(Speed=('Speed', 'mean'), 
                                is_weekend=('is_weekend', 'first')).reset_index() 
wd = day_avg[~day_avg['is_weekend']]['Speed']
we = day_avg[day_avg['is_weekend']]['Speed']

t, p = stats.ttest_ind(we, wd, equal_var=____)
print('평일 평균:', round(wd.mean(), 2), f'(n={len(wd)})')
print('주말 평균:', round(we.mean(), 2), f'(n={len(we)})')
print(f't통계량: {t:.3f}, p값: {p:.2e}')

## 9. 요일별 평균 속도와 신뢰구간

In [ ]:
names = ['월', '화', '수', '목', '금', '토', '일']
p2 = df.copy()
p2['요일'] = p2['dow'].map(dict(enumerate(names))) 
fig, ax = plt.subplots(figsize=(5, 3))
sns.pointplot(data=p2, x='요일', 
              y='Speed', order=names,
              errorbar=('ci', ____), capsize=.1)
plt.title('요일별 평균 속도와 95% 신뢰구간')
plt.show()

## 10. 일별 평균 속도의 이상 날짜 탐색

In [ ]:
x = daily_mean.copy()
x['z']=(x.Speed-x.Speed.mean())/x.Speed.std(ddof=1) 
colors = np.where(x.z.abs() >= ____, 'C1', 'C0')

plt.figure(figsize=(8, 4))
plt.bar(x.Date, x.Speed, color=colors)
plt.title('일별 평균 속도와 이상 날짜 탐색')
plt.show()

## 11. 후보 날짜 확인하기

In [ ]:
outliers = x.loc[x.z.abs() >= ____, ['Date', 'Speed', 'z']].copy()
outliers[['Speed', 'z']] = outliers[['Speed', 'z']].round(2)
print(outliers.to_string(index=False))

## 12. 통행량의 7일 이동평균

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))

daily_mean['ToVol_7d'] = daily_mean['ToVol'].rolling(____, min_periods=1).mean()
ax.plot(daily_mean.Date, daily_mean.ToVol, alpha=.35, marker='o', label='일별 총통행량')
ax.plot(daily_mean.Date, daily_mean.ToVol_7d, lw=2.5, label='7일 이동평균')
ax.legend(); ax.set_title('일별 통행량과 7일 이동평균'); plt.show()

## 심화 탐구 — 빈칸 정답

PPT의 빈칸 순서대로 입력하세요.

1. `OccRate`
2. `OccRate`
3. `OccRate`
4. `we2`
5. `wd2`

1번부터 3번까지는 PPT의 따옴표 안에 입력합니다.